In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/train.csv")
df.head()

,created_date,post_id,emoticon_1,emoticon_2,emoticon_3,upvote,downvote,if_1,if_2,race,religion,gender,disability,comment,label
0,2024-01-18 08:43:57.397508+00:00,73,0,0,0,0,1,0,10,NaN,NaN,NaN,False,She might be a bright spot for a party keou on...,2
1,2024-03-24 21:43:11.490017+00:00,39,0,0,0,6,0,0,4,NaN,NaN,NaN,False,"Under Alaska law, a non-tribal member is not b...",0
2,2024-04-24 20:32:17.014931+00:00,31,0,1,1,0,0,0,10,NaN,NaN,NaN,False,in the future please spare me your strawman dr...,2
3,2023-05-28 22:00:14.214527+00:00,39,0,0,0,5,0,0,10,NaN,NaN,NaN,False,"PS: That should have been ""rot"" instead of ""co...",2
4,2023-09-09 23:12:05.689498+00:00,39,0,0,0,0,0,0,10,NaN,NaN,NaN,False,"Today, the confederate flag...tomorrow, the na...",2


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198000 entries, 0 to 197999
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   created_date  198000 non-null  object
 1   post_id       198000 non-null  int64 
 2   emoticon_1    198000 non-null  int64 
 3   emoticon_2    198000 non-null  int64 
 4   emoticon_3    198000 non-null  int64 
 5   upvote        198000 non-null  int64 
 6   downvote      198000 non-null  int64 
 7   if_1          198000 non-null  int64 
 8   if_2          198000 non-null  int64 
 9   race          52577 non-null   object
 10  religion      52577 non-null   object
 11  gender        52577 non-null   object
 12  disability    198000 non-null  bool  
 13  comment       197999 non-null  object
 14  label         198000 non-null  int64 
dtypes: bool(1), int64(9), object(5)
memory usage: 21.3+ MB


Split the train dataset using train_test_split with random_state=42 such that 80% is training data and remaining 20% is validation data. store the data as X_train, X_val, y_train, y_val. Lets say training data (X_train) has shape (a,b) where a is number of rows and b is the number of features, similarly validation set (X_val) has shape (c,d) where c is the number of rows and d is the number of features. what will be the value of a + c ?

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['label'])
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

a = X_train.shape[0]
c = X_val.shape[0]

print("a + c =", a + c)   

a + c = 198000


After converting created_date to a datetime object and extracting the day, month, and year for your X_train and X_val created in the first question, identify the most frequently occurring month across all years in X_train. Which month is it?

In [5]:
X_train_dt = X_train.copy()

X_train_dt['created_date'] = pd.to_datetime(X_train_dt['created_date'], errors='coerce')
X_train_dt['month'] = X_train_dt['created_date'].dt.month

most_common_month = X_train_dt['month'].mode()[0]

print("Most frequent month:", most_common_month)

Most frequent month: 5


Impute the null values of categorical features with the value 'none'. Encode only religion', 'gender', 'race' features using one hot encoding, setting handle_unknown='ignore'. Make sure the output is a pandas dataframe. Let's say the shape of X_train after imputing is (a,b), what is the value of b ?
Note: make sure to transform X_val as well using one hot encoding.

In [6]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['religion', 'gender', 'race']

imputer = SimpleImputer(strategy='constant', fill_value='none')
X_train_cat = imputer.fit_transform(X_train[cat_cols])
X_val_cat = imputer.transform(X_val[cat_cols])

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_ohe = encoder.fit_transform(X_train_cat)
X_val_ohe = encoder.transform(X_val_cat)

X_train_ohe_df = pd.DataFrame(
    X_train_ohe, columns=encoder.get_feature_names_out(cat_cols)
)

print("Number of columns after encoding:", X_train_ohe_df.shape[1])  # ✅ b

Number of columns after encoding: 19


Apply CountVectorizer to the column comment of X_train and X_val obtained in previous questions, what is the sum of the counts for the document at index 1 (the second row) of the transformed X_train sparse matrix?
Note: Please ensure to transform X_val as well.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_text = vectorizer.fit_transform(
    X_train['comment'].fillna('')
)
X_val_text = vectorizer.transform(
    X_val['comment'].fillna('')
)

row_sum = X_train_text[1].sum()

print("Sum of counts for document index 1:", row_sum)  # ✅ ANSWER

Sum of counts for document index 1: 23


Convert 'disability' feature into integer type, with True being mapped to 1 and False mapped to 0. What is the sum of all the 'disability values in X_train and X_val after the transformation?

In [8]:
X_train_dis = X_train.copy()
X_val_dis = X_val.copy()

X_train_dis['disability'] = X_train_dis['disability'].astype(int)
X_val_dis['disability'] = X_val_dis['disability'].astype(int)

total_sum = X_train_dis['disability'].sum() + X_val_dis['disability'].sum()

print("Sum of disability values:", total_sum)  # ✅ ANSWER

Sum of disability values: 2743


Scale the numeric features using StandardScaler. What are the number of features seen during fit for X_train? Ensure that you drop all datetime columns first.

In [9]:
from sklearn.preprocessing import StandardScaler

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
num_cols = num_cols.drop(['created_date'], errors='ignore')

scaler = StandardScaler()
scaler.fit(X_train[num_cols])

print("Number of features seen:", scaler.n_features_in_)  # ✅ ANSWER

Number of features seen: 8


 Preprocess the data in the following manner and train the model:

0) Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms X_val, y_val.

After splitting:

Impute missing values by fitting the SimpleImputer on X_train. You are free to use any valid strategy for Imputation.

Use the fitted imputer to transform both X_train and X_val

Ensure that after imputation no value in X_train or X_val is negative.

If any negative value exists, convert it to positive using the absolute value function

1) Convert the created_date feature into a datetime object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.

2) Transform the "comment" feature using TfidfVectorizer with stop_words='english'. Make sure to transform X_val accordingly.

3) Encode all the categorical features using one hot encoder with handle_unknown='ignore'. Make sure to transform X_val accordingly.
Now train a Multinomial Naive Bayes model on the preprocessed data. Use a pipeline if you need to. What is the macro F1 score obtained on the train dataset?


Note: you may train Multinomial Naive Bayes model on the preprocessed data which has been obtained from previous questions and give the answer, both ways the answers will be accepted. 

If some feature is creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

What is the macro F1 score obtained on the validation dataset?

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

df_nb = df.copy()

X = df_nb.drop(columns=['label'])
y = df_nb['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train['comment'] = X_train['comment'].fillna('')
X_val['comment'] = X_val['comment'].fillna('')

# Date features
for d in [X_train, X_val]:
    d['created_date'] = pd.to_datetime(d['created_date'], errors='coerce')
    d['day'] = d['created_date'].dt.day
    d['month'] = d['created_date'].dt.month
    d['year'] = d['created_date'].dt.year
    d.drop(columns=['created_date'], inplace=True)

text_features = 'comment'
cat_features = X_train.select_dtypes(include='object').columns.drop('comment')
num_features = X_train.select_dtypes(include=['int64','float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english'), text_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', MultinomialNB())
])

pipeline.fit(X_train, y_train)

train_f1 = f1_score(y_train, pipeline.predict(X_train), average='macro')
val_f1 = f1_score(y_val, pipeline.predict(X_val), average='macro')

print("Train Macro F1:", train_f1)
print("Validation Macro F1:", val_f1)

Train Macro F1: 0.4876068222225023
Validation Macro F1: 0.4742713786141589


Process the data in the following manner as given below and train the model.
0) Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms X_val, y_val.
After splitting:

Impute missing values by fitting the SimpleImputer on X_train. You are free to use any valid strategy for Imputation.

Use the fitted imputer to transform both X_train and X_val

Ensure that after imputation no value in X_train or X_val is negative.

If any negative value exists, convert it to positive using the absolute value function

1) Convert the created_date feature into a datetime object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.

2) Create a new binary categorical column named is_weekend where Saturday and Sunday are represented as 1 and all other days as 0. 

3) Apply TfidfVectorizer(stop_words='english') to the comment column. Make sure to transform X_val accordingly.

4) Apply OneHotEncoder(handle_unknown='ignore') to the categorical features including newly created categorical feature is_weekend. Make sure to transform X_val accordingly.

After all above transformation, train MultinomialNB model.

After training, use the model to predict the labels for X_train and calculate the Macro F1 score. What is the resulting score rounded to four decimal places?

note: If some features are creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).
*


In [11]:
df_nb2 = df.copy()

X = df_nb2.drop(columns=['label'])
y = df_nb2['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train['comment'] = X_train['comment'].fillna('')
X_val['comment'] = X_val['comment'].fillna('')

for d in [X_train, X_val]:
    d['created_date'] = pd.to_datetime(d['created_date'], errors='coerce')
    d['day'] = d['created_date'].dt.day
    d['month'] = d['created_date'].dt.month
    d['year'] = d['created_date'].dt.year
    d['is_weekend'] = d['created_date'].dt.weekday.isin([5,6]).astype(int)
    d.drop(columns=['created_date'], inplace=True)

text_features = 'comment'
cat_features = X_train.select_dtypes(include='object').columns.drop('comment')
cat_features = list(cat_features) + ['is_weekend']
num_features = X_train.select_dtypes(include=['int64','float64']).columns.drop('is_weekend')

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english'), text_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', MultinomialNB())
])

pipeline.fit(X_train, y_train)

train_f1 = f1_score(y_train, pipeline.predict(X_train), average='macro')
val_f1 = f1_score(y_val, pipeline.predict(X_val), average='macro')

print("Train Macro F1:", round(train_f1,4))   # ✅ ANSWER
print("Validation Macro F1:", round(val_f1,4)) # ✅ ANSWER

Train Macro F1: 0.4887
Validation Macro F1: 0.4754
